# Jain 2026 — MAGeCK processing

MAGeCK `test` and `pathway` analysis for the Jain 2026 in vivo CAR-T CRISPR screen.

All paths are relative to the directory containing this notebook, which is
assumed to sit one level below the project root:

    <project root>/
    |-- <directory containing this notebook>/
    |   `-- Jain_2026.ipynb
    |-- Jain_2026_Screen27/
    |   |-- Raw/                    # source counts (.xlsx), NTC list, edited counts
    |   |-- XC_Mageck/              # mageck test output
    |   `-- Jain_2026_For_Kevin/    # curated copies for handoff
    `-- Pathway_Analysis/
        `-- h.all.v2024.1.Hs.symbols.gmt

Requires `mageck` on `PATH` (v0.5.9.5 was used for the stored outputs).

In [ ]:
import os
import shutil
import itertools
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Working directory:", os.getcwd())

In [ ]:
# reshape the count files
# naive
df = pd.read_excel("../Jain_2026_Screen27/Raw/Sadelain_In vivo CRISPR screen_sgRNA counts_.xlsx")
keep_cols = ["sgRNA", "Gene", "baseline_1", "baseline_2", "baseline_3", "Day20_1", "Day20_2", "Day20_3"]
df = df[keep_cols]

def rename_non_targeting_genes(gene_name):
    if ("HumanControl" in gene_name):
        return "NonTargetingControl"
    return gene_name

df['Gene'] = df['Gene'].apply(rename_non_targeting_genes)

df.to_csv("../Jain_2026_Screen27/Raw/counts_edited.txt", sep="\t", index=False)

In [ ]:
ntc_list = df[df['Gene'].str.contains("NonTargetingControl", na=False)]['sgRNA']
print(ntc_list.shape[0])
output_file = "../Jain_2026_Screen27/Raw/NTC_list.txt"
ntc_list.to_csv(output_file, index=False, header=False)

print(f"Non-targeting control sgRNA list saved to {output_file}")

218
Non-targeting control sgRNA list saved to ../Jain_2026_Screen27/Raw/NTC_list.txt


In [ ]:
os.makedirs("../Jain_2026_Screen27/XC_Mageck", exist_ok=True)

In [ ]:
control_sgrna_file = "../Jain_2026_Screen27/Raw/NTC_list.txt"
count_file = "../Jain_2026_Screen27/Raw/counts_edited.txt"
mageck_dir = "../Jain_2026_Screen27/XC_Mageck"

mapping = {
    "Input": ["baseline_1", "baseline_2", "baseline_3"],
    "Day20_CAR-T": ["Day20_1", "Day20_2", "Day20_3"]
}

def run_mageck(count_file, mapping_dict):
    for control, treatment in itertools.combinations(mapping_dict.keys(), 2):

        control_reps = ",".join(mapping_dict[control])
        treatment_reps = ",".join(mapping_dict[treatment])

        output_prefix = os.path.join(mageck_dir, f"{treatment}_vs_{control}")

        cmd = (
            f"mageck test -k {count_file} "
            f"-t {treatment_reps} -c {control_reps} "
            f"--control-sgrna {control_sgrna_file} "
            f"--norm-method control "
            f"-n {output_prefix}"
        )

        print(cmd)
        os.system(cmd)

        for suffix in ["sgrna_summary.txt", "gene_summary.txt"]:
            out_file = f"{output_prefix}.{suffix}"
            if os.path.exists(out_file):
                df_out = pd.read_csv(out_file, sep="\t")

                if "Input" in control:
                    df_out["Cell_Type"] = treatment
                elif "Input" in treatment:
                    df_out["Cell_Type"] = control
                else:
                    df_out["Cell_Type_1"] = treatment
                    df_out["Cell_Type_2"] = control

                df_out.to_csv(out_file, sep="\t", index=False)


run_mageck(count_file, mapping)

print("MAGeCK analysis completed.")

In [ ]:
input_folders = [
    "../Jain_2026_Screen27/Jain_2026_For_Kevin/cell_vs_input"
]

gmt_file = "../Pathway_Analysis/h.all.v2024.1.Hs.symbols.gmt"

for input_folder in input_folders:
    output_folder = os.path.join(os.path.dirname(input_folder), os.path.basename(input_folder) + "_pathway")
    os.makedirs(output_folder, exist_ok=True)

    for file in os.listdir(input_folder):
        if file.endswith("gene_summary.txt"):
            input_file = os.path.join(input_folder, file)
            base_name = file.replace(".gene_summary.txt", "")
            output_prefix = os.path.join(output_folder, base_name)

            gene_summary = pd.read_csv(input_file, sep="\t")
            cell_type_columns = [col for col in gene_summary.columns if "Cell_Type" in col]
            cell_type_data = gene_summary[cell_type_columns] if cell_type_columns else None

            command = ["mageck", "pathway", "--gene-ranking", input_file, "--gmt", gmt_file, "--output-prefix", output_prefix]

            try:
                print(f"Running MAGeCK pathway for: {file}")
                subprocess.run(command, check=True)

                pathway_output_file = f"{output_prefix}.pathway_summary.txt"

                if os.path.exists(pathway_output_file) and cell_type_data is not None:
                    pathway_results = pd.read_csv(pathway_output_file, sep="\t")
                    updated_results = pd.concat([pathway_results, cell_type_data], axis=1)
                    updated_results.to_csv(pathway_output_file, sep="\t", index=False)
            except subprocess.CalledProcessError as e:
                print(f"Error running MAGeCK pathway for {file}: {e}")

print("MAGeCK pathway analysis completed for all files.")